# 单张图片两阶段推理

这个 notebook 用于对单张图片执行两阶段推理：
- 第一阶段：绝缘子检测模型输出 `detections` 和数量
- 第二阶段：按 bbox 裁剪实例并用分类模型判断 `normal / abnormal`

最终会输出：
- `overview/`：原图总览结果图
- `instances/`：每个实例单独结果图
- `contact_sheet/`：实例拼图
- `abnormal_crops/`：异常实例原始 crop
- `summary.json`


## 0. 依赖导入


In [ ]:
from pathlib import Path
import json
from collections import Counter

from PIL import Image, ImageDraw
from ultralytics import YOLO

print('✓ 依赖导入完成')


## 1. 路径与参数配置


In [ ]:
DET_MODEL_PATH = Path('/home/hujing/yolo-web-ui/experiments/detector/weights/best.pt')
CLS_MODEL_PATH = Path('/home/hujing/yolo-web-ui/experiments_cls/insulator_cls_exp001/weights/best.pt')
IMAGE_PATH = Path('/home/hujing/yolo-web-ui/offline_workspace/datasets/data_annotated_2class/DJI_0175_W.JPG')
OUTPUT_DIR = Path('/home/hujing/yolo-web-ui/offline_workspace/outputs/two_stage_inference')

DET_CONF = 0.25
DET_IOU = 0.45
CROP_PADDING = 0.02
CONTACT_SHEET_TILE_SIZE = 240

OVERVIEW_DIR = OUTPUT_DIR / 'overview'
INSTANCES_DIR = OUTPUT_DIR / 'instances'
CONTACT_SHEET_DIR = OUTPUT_DIR / 'contact_sheet'
ABNORMAL_CROPS_DIR = OUTPUT_DIR / 'abnormal_crops'

for directory in [OUTPUT_DIR, OVERVIEW_DIR, INSTANCES_DIR, CONTACT_SHEET_DIR, ABNORMAL_CROPS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print('DET_MODEL_PATH =', DET_MODEL_PATH)
print('CLS_MODEL_PATH =', CLS_MODEL_PATH)
print('IMAGE_PATH =', IMAGE_PATH)
print('OUTPUT_DIR =', OUTPUT_DIR)


## 2. 加载模型


In [ ]:
det_model = YOLO(str(DET_MODEL_PATH))
cls_model = YOLO(str(CLS_MODEL_PATH))

print('✓ 检测模型加载完成')
print('✓ 分类模型加载完成')


## 3. 工具函数


In [ ]:
def clamp_bbox(bbox, width, height):
    x1, y1, x2, y2 = bbox
    x1 = max(0, min(int(round(x1)), width - 1))
    y1 = max(0, min(int(round(y1)), height - 1))
    x2 = max(x1 + 1, min(int(round(x2)), width))
    y2 = max(y1 + 1, min(int(round(y2)), height))
    return [x1, y1, x2, y2]


def add_padding_to_bbox(bbox, width, height, padding_ratio=0.02):
    x1, y1, x2, y2 = bbox
    box_w = max(1, x2 - x1)
    box_h = max(1, y2 - y1)
    pad_x = box_w * padding_ratio
    pad_y = box_h * padding_ratio
    return clamp_bbox([x1 - pad_x, y1 - pad_y, x2 + pad_x, y2 + pad_y], width, height)


def detect_insulators(model, image_path, conf=0.25, iou=0.45):
    result = model.predict(source=str(image_path), conf=conf, iou=iou, verbose=False)[0]
    names = result.names if hasattr(result, 'names') else {}
    detections = []

    if result.boxes is None:
        return detections

    for index, box in enumerate(result.boxes):
        xyxy = box.xyxy[0].tolist()
        cls_id = int(box.cls.item()) if box.cls is not None else 0
        conf_score = float(box.conf.item()) if box.conf is not None else 0.0
        detections.append({
            'id': index,
            'bbox': xyxy,
            'det_class_id': cls_id,
            'det_class_name': names.get(cls_id, f'class_{cls_id}'),
            'det_confidence': conf_score,
        })

    return detections


def classify_crop(model, crop_image):
    result = model.predict(source=crop_image, verbose=False)[0]
    probs = result.probs
    top1 = int(probs.top1)
    top1_conf = float(probs.top1conf.item())
    class_name = result.names.get(top1, f'class_{top1}')
    scores = {
        result.names.get(i, f'class_{i}'): float(score)
        for i, score in enumerate(probs.data.tolist())
    }
    return {
        'class_id': top1,
        'class_name': class_name,
        'confidence': top1_conf,
        'scores': scores,
    }


def draw_results(image, predictions):
    rendered = image.copy()
    draw = ImageDraw.Draw(rendered)
    for item in predictions:
        x1, y1, x2, y2 = item['bbox']
        label = item['classification']['class_name']
        score = item['classification']['confidence']
        det_id = item['id']
        color = 'red' if label == 'abnormal' else 'green'
        draw.rectangle([x1, y1, x2, y2], outline=color, width=4)
        draw.text((x1 + 4, max(0, y1 - 18)), f"#{det_id} {label} {score:.2f}", fill=color)
    return rendered


def render_instance_image(crop, item):
    canvas = crop.copy()
    draw = ImageDraw.Draw(canvas)
    label = item['classification']['class_name']
    score = item['classification']['confidence']
    det_id = item['id']
    color = 'red' if label == 'abnormal' else 'green'
    draw.rectangle([0, 0, canvas.width - 1, canvas.height - 1], outline=color, width=4)
    draw.text((8, 8), f"#{det_id} {label} {score:.2f}", fill=color)
    return canvas


def fit_crop_to_tile(crop, tile_size=240, background_color='white'):
    canvas = Image.new('RGB', (tile_size, tile_size), background_color)
    fitted = crop.copy()
    fitted.thumbnail((tile_size, tile_size))
    offset_x = (tile_size - fitted.width) // 2
    offset_y = (tile_size - fitted.height) // 2
    canvas.paste(fitted, (offset_x, offset_y))
    return canvas


def build_contact_sheet(predictions, tile_size=240):
    if not predictions:
        return Image.new('RGB', (400, 200), 'white')

    header_height = 28
    cols = min(4, len(predictions))
    rows = (len(predictions) + cols - 1) // cols
    sheet = Image.new('RGB', (cols * tile_size, rows * (tile_size + header_height)), 'white')

    for index, item in enumerate(predictions):
        tile = fit_crop_to_tile(item['crop'], tile_size=tile_size)
        offset_x = (index % cols) * tile_size
        offset_y = (index // cols) * (tile_size + header_height)
        sheet.paste(tile, (offset_x, offset_y + header_height))
        draw = ImageDraw.Draw(sheet)
        label = item['classification']['class_name']
        score = item['classification']['confidence']
        draw.text((offset_x + 8, offset_y + 4), f"#{item['id']} {label} {score:.2f}", fill='black')

    return sheet


## 4. 第一阶段检测


In [ ]:
image = Image.open(IMAGE_PATH).convert('RGB')
image_width, image_height = image.size

detections = detect_insulators(det_model, IMAGE_PATH, conf=DET_CONF, iou=DET_IOU)
print('检测到绝缘子数量 =', len(detections))
detections[:3]


## 5. 第二阶段分类


In [ ]:
predictions = []

for det in detections:
    padded_bbox = add_padding_to_bbox(det['bbox'], image_width, image_height, CROP_PADDING)
    x1, y1, x2, y2 = padded_bbox
    crop = image.crop((x1, y1, x2, y2))
    cls_result = classify_crop(cls_model, crop)

    record = {
        'id': det['id'],
        'bbox': padded_bbox,
        'detection': det,
        'classification': cls_result,
        'crop': crop,
    }
    predictions.append(record)

    instance_label = cls_result['class_name']
    instance_path = INSTANCES_DIR / f"{IMAGE_PATH.stem}_inst{det['id']:03d}_{instance_label}.jpg"
    render_instance_image(crop, record).save(instance_path)

    if cls_result['class_name'] == 'abnormal':
        crop_path = ABNORMAL_CROPS_DIR / f"{IMAGE_PATH.stem}_abnormal_{det['id']:03d}.jpg"
        crop.save(crop_path)

print('完成分类实例数 =', len(predictions))
predictions[:3]


## 6. 汇总统计


In [ ]:
class_counter = Counter(item['classification']['class_name'] for item in predictions)
abnormal_items = [item for item in predictions if item['classification']['class_name'] == 'abnormal']

summary = {
    'image_path': str(IMAGE_PATH),
    'det_model_path': str(DET_MODEL_PATH),
    'cls_model_path': str(CLS_MODEL_PATH),
    'detection_count': len(detections),
    'class_counts': dict(class_counter),
    'abnormal_count': class_counter.get('abnormal', 0),
    'normal_count': class_counter.get('normal', 0),
    'abnormal_instances': [
        {
            'id': item['id'],
            'bbox': item['bbox'],
            'confidence': item['classification']['confidence'],
            'scores': item['classification']['scores'],
        }
        for item in abnormal_items
    ],
    'predictions': [
        {
            'id': item['id'],
            'bbox': item['bbox'],
            'detection': item['detection'],
            'classification': item['classification'],
        }
        for item in predictions
    ],
}

print(json.dumps({
    'detection_count': summary['detection_count'],
    'normal_count': summary['normal_count'],
    'abnormal_count': summary['abnormal_count'],
}, ensure_ascii=False, indent=2))


## 7. 可视化与导出


In [ ]:
overview_image = draw_results(image, predictions)
contact_sheet_image = build_contact_sheet(predictions, tile_size=CONTACT_SHEET_TILE_SIZE)

overview_path = OVERVIEW_DIR / f"{IMAGE_PATH.stem}_overview.jpg"
contact_sheet_path = CONTACT_SHEET_DIR / f"{IMAGE_PATH.stem}_sheet.jpg"
summary_path = OUTPUT_DIR / 'summary.json'

overview_image.save(overview_path)
contact_sheet_image.save(contact_sheet_path)
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

print('overview 已保存到:', overview_path)
print('contact_sheet 已保存到:', contact_sheet_path)
print('instances 已保存到:', INSTANCES_DIR)
print('abnormal_crops 已保存到:', ABNORMAL_CROPS_DIR)
print('summary.json 已保存到:', summary_path)
overview_image
